# WP3b: counterfactual image conditions for ONE trained adapter (set ADAPTER_RUN in Cell B)
GPU on, Internet on, secrets GITHUB_TOKEN + HF_TOKEN attached. Reuses the zero-shot (base) conditions saved earlier under wp3_full on the Hub, so it only runs the adapter (about 1 hour). First do a quick test commit with N_EVAL = 20.

In [ ]:
# CELL A - INSTALL (first cell; nothing heavy imported before it). Stops immediately if an install fails.
import sys, subprocess
print("Python", sys.version.split()[0])
def pip(args, must=True):
    r = subprocess.run(f"{sys.executable} -m pip {args}", shell=True, capture_output=True, text=True)
    if r.returncode != 0 and must:
        raise RuntimeError("pip failed: " + args + "\n" + (r.stdout + r.stderr)[-2500:])
    return r
# No numpy/pandas here on purpose: upgrading numpy inside a running kernel breaks it. qwen-vl-utils without the decord extra (video only).
pip("install -q -U transformers accelerate peft 'qwen-vl-utils==0.0.8' datasets huggingface_hub matplotlib tabulate")
pip("install -q --force-reinstall 'pillow<11'")
pip("uninstall -y torchao", must=False)      # Kaggle's old torchao breaks newer peft; not used here
print("Install done.")

In [ ]:
# CELL B - settings, secrets, Hugging Face login, GitHub clone (token never stays in .git/config)
import os, subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, login

# ---------------- SETTINGS ----------------
ADAPTER_RUN  = "slake_img25_s42"   # which trained adapter to analyze: any run id under runs/ on the Hub (e.g. slake_img5_s42, slake_img25_s42)
N_EVAL       = None                # None = all 1,061 test questions. Use 20 for a quick test commit.
HF_REPO_NAME = "med-vqa-qwen2.5-vl-slake"
GITHUB_REPO  = "sanshubhGupta/med-vqa-lora-btp"
TEST = N_EVAL is not None
RUN_TAG = f"wp3b_{ADAPTER_RUN}" + ("_test" if TEST else "")
# ------------------------------------------

GH_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(GH_TOKEN, "***")[-2000:])
REPO = Path("/kaggle/working/med-vqa-lora-btp")
sh('git config --global user.email "sanshubh08gupta@gmail.com"'); sh('git config --global user.name "sanshubhGupta"')
if not REPO.exists():
    sh(f"cd /kaggle/working && git clone https://{GH_TOKEN}@github.com/{GITHUB_REPO}.git")
    sh(f"cd {REPO} && git remote set-url origin https://github.com/{GITHUB_REPO}.git")

login(token=UserSecretsClient().get_secret("HF_TOKEN"), add_to_git_credential=False)
hf_api = HfApi(); HF_REPO_ID = f"{hf_api.whoami()['name']}/{HF_REPO_NAME}"
_files = set(hf_api.list_repo_files(HF_REPO_ID))
for need in (f"runs/{ADAPTER_RUN}/adapter/adapter_model.safetensors", f"runs/{ADAPTER_RUN}/results/predictions.csv",
             "results_full/predictions_baseline_zero_shot.csv", "wp3_full/predictions_base_blank.csv",
             "wp3_full/predictions_base_shuffled.csv", "wp3_full/predictions_base_noimage.csv"):
    assert need in _files, f"missing on the Hub: {need}"
print("HF OK:", HF_REPO_ID, "| analyzing adapter:", ADAPTER_RUN, "| mode:", "QUICK TEST" if TEST else "FULL RUN")

def push_folder(folder, path_in_repo=None, msg="Upload"):
    try:
        hf_api.upload_folder(folder_path=str(folder), repo_id=HF_REPO_ID, repo_type="model", path_in_repo=path_in_repo, commit_message=msg)
        print("Pushed to", f"https://huggingface.co/{HF_REPO_ID}", f"({path_in_repo})" if path_in_repo else "")
    except Exception as e:
        print("HF push FAILED (continuing):", repr(e)[:300])

In [ ]:
# CELL C - data, and the three counterfactual image conditions
import gc, json, random, re, time, zipfile
import numpy as np, pandas as pd, torch
from PIL import Image
from datasets import load_dataset
from huggingface_hub import snapshot_download, hf_hub_download

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
PROMPT   = "Answer the question in a few words only. Question: {q}"          # identical to training
MAX_PIX, MIN_PIX = 256 * 28 * 28, 64 * 28 * 28                              # identical to training
OUT = Path(f"/kaggle/working/{RUN_TAG}"); OUT.mkdir(parents=True, exist_ok=True)
random.seed(42); np.random.seed(42); torch.manual_seed(42)

slake = load_dataset("BoKelvin/SLAKE")
slake_dir = Path(snapshot_download(repo_id="BoKelvin/SLAKE", repo_type="dataset"))
extract_dir = slake_dir / "unzipped_imgs"
if not extract_dir.exists():
    with zipfile.ZipFile(slake_dir / "imgs.zip") as z:
        z.extractall(extract_dir)
def get_img(row):
    return Image.open(Path(extract_dir) / "imgs" / row["img_name"]).convert("RGB")

train_en = slake["train"].filter(lambda r: r["q_lang"] == "en")
test_en  = slake["test"].filter(lambda r: r["q_lang"] == "en")
n_eval = len(test_en) if N_EVAL is None else min(N_EVAL, len(test_en))
img_names = list(test_en["img_name"])
print("English test:", len(test_en), "| evaluating", n_eval, "| test images:", len(set(img_names)))

# Shuffled-image condition: each question gets an image from a DIFFERENT test image (fixed seed)
_rng = np.random.RandomState(42); partner = np.zeros(len(test_en), dtype=int)
for i in range(len(test_en)):
    while True:
        j = _rng.randint(0, len(test_en))
        if img_names[j] != img_names[i]: break
    partner[i] = j

def make_image(i, cond):
    """cond: blank = gray image of the same size; shuffled = another test image; noimage = no image at all."""
    if cond == "noimage": return None
    real = get_img(test_en[i])
    if cond == "blank": return Image.new("RGB", real.size, (128, 128, 128))
    if cond == "shuffled": return get_img(test_en[int(partner[i])])
    raise ValueError(cond)

In [ ]:
# CELL D - load base model, attach the trained adapter, and define evaluation per condition
from contextlib import nullcontext
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from peft import PeftModel

processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=MIN_PIX, max_pixels=MAX_PIX)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
model = PeftModel.from_pretrained(model, HF_REPO_ID, subfolder=f"runs/{ADAPTER_RUN}/adapter"); model.eval()
print("adapter attached:", ADAPTER_RUN)

def build_inputs(image, question):
    content = [{"type": "text", "text": PROMPT.format(q=question)}]
    if image is not None:
        content = [{"type": "image", "image": image, "min_pixels": MIN_PIX, "max_pixels": MAX_PIX}] + content
    msgs = [{"role": "user", "content": content}]
    prompt_text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    if image is None:
        return processor(text=[prompt_text], return_tensors="pt")
    imgs, vids = process_vision_info(msgs)
    return processor(text=[prompt_text], images=imgs, videos=vids, return_tensors="pt")

def norm(s):
    s = re.sub(r"[^\w\s]", " ", str(s).lower().strip())
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    toks = [t[:-1] if len(t) > 3 and t.endswith("s") else t for t in s.split()]
    return " ".join(toks)

def score(pred, gt):
    p, g = norm(pred), norm(gt)
    gt_tok, pred_tok = set(g.split()), set(p.split())
    return p == g, (len(gt_tok & pred_tok) / len(gt_tok) if gt_tok else 0.0)

@torch.no_grad()
def predict(image, question):
    inputs = build_inputs(image, question).to(model.device)
    out = model.generate(**inputs, max_new_tokens=32, do_sample=False)
    return processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()

def evaluate_cond(mtag, cond):
    f = OUT / f"predictions_{mtag}_{cond}.csv"
    if f.exists():
        print("found", f.name, "- loading"); return pd.read_csv(f, keep_default_na=False, dtype={"gt": str, "pred": str, "question": str})
    rows, t0 = [], time.time()
    for i in range(n_eval):
        r = test_en[i]
        pred = predict(make_image(i, cond), r["question"])
        exact, rec = score(pred, r["answer"])
        rows.append(dict(i=i, img_name=r["img_name"], question=r["question"], gt=str(r["answer"]), pred=pred,
                         answer_type=r["answer_type"], content_type=r["content_type"], exact=exact, recall=rec))
        if (i + 1) % 100 == 0:
            print(f"[{mtag}/{cond}] {i + 1}/{n_eval} ({(time.time() - t0) / 60:.1f} min)")
    d = pd.DataFrame(rows); d.to_csv(f, index=False)
    print(f"[{mtag}/{cond}] exact={d.exact.mean():.4f} (closed {d[d.answer_type=='CLOSED'].exact.mean():.4f}, open {d[d.answer_type=='OPEN'].exact.mean():.4f})")
    return d

In [ ]:
# CELL E - run the counterfactual conditions for THIS adapter only (base-model conditions are reused from wp3_full on the Hub)
for cond in ("blank", "shuffled", "noimage"):
    evaluate_cond("lora", cond)
    push_folder(OUT, path_in_repo=f"wp3b/{ADAPTER_RUN}", msg=f"wp3b {ADAPTER_RUN}/{cond}")
print("All conditions done.")

In [ ]:
# CELL F - analysis: accuracy by image condition, gain decomposition (blank AND other-image counterfactuals), sensitivity, answer format, lookup baseline
import matplotlib.pyplot as plt

def _per_image(df, value, mask=None):
    imgs = df["img_name"].unique(); x = df if mask is None else df[mask]
    g = x.groupby("img_name")[value].agg(["sum", "count"]).reindex(imgs, fill_value=0)
    return g["sum"].values.astype(float), g["count"].values.astype(float)
def cluster_boot_mean(df, value, mask=None, n_boot=5000, seed=0):
    S, N = _per_image(df, value, mask); rng = np.random.RandomState(seed)
    idx = rng.randint(0, len(S), (n_boot, len(S))); num, den = S[idx].sum(1), N[idx].sum(1); est = num[den > 0] / den[den > 0]
    return (S.sum() / N.sum() if N.sum() else float("nan")), np.percentile(est, 2.5), np.percentile(est, 97.5), int(N.sum())
def rd(path):
    p = hf_hub_download(HF_REPO_ID, path, repo_type="model")
    return pd.read_csv(p, keep_default_na=False, dtype={"gt": str, "pred": str, "question": str}).iloc[:n_eval].reset_index(drop=True)

conds = ("blank", "shuffled", "noimage")
parts = {("base", "real"): rd("results_full/predictions_baseline_zero_shot.csv"), ("lora", "real"): rd(f"runs/{ADAPTER_RUN}/results/predictions.csv")}
for c in conds:
    parts[("base", c)] = rd(f"wp3_full/predictions_base_{c}.csv")
    parts[("lora", c)] = pd.read_csv(OUT / f"predictions_lora_{c}.csv", keep_default_na=False, dtype={"gt": str, "pred": str, "question": str})
d = pd.DataFrame({"img_name": img_names[:n_eval], "answer_type": parts[("base", "real")]["answer_type"].values,
                  "content_type": parts[("base", "real")]["content_type"].values, "gt": parts[("base", "real")]["gt"].values})
for (m, c), p in parts.items():
    assert len(p) == n_eval and (p["gt"].values == d["gt"].values).all(), f"row mismatch in {m}/{c}"
    d[f"ex_{m}_{c}"] = p["exact"].astype(float).values; d[f"pr_{m}_{c}"] = p["pred"].values

def ci(col, mask=None):
    pt, lo, hi, n = cluster_boot_mean(d, col, mask); return f"{pt:.3f} [{lo:.3f}, {hi:.3f}]", n
subsets = [("overall", None), ("CLOSED", d.answer_type == "CLOSED"), ("OPEN", d.answer_type == "OPEN")] + [(c, d.content_type == c) for c in sorted(d.content_type.unique())]

rows = []
for name, mask in subsets:
    r = {"subset": name}
    for m in ("base", "lora"):
        for c in ("real",) + conds: r[f"{m}_{c}"], n = ci(f"ex_{m}_{c}", mask)
    r["n"] = n; rows.append(r)
acc = pd.DataFrame(rows); acc.to_csv(OUT / "accuracy_by_condition.csv", index=False)
print(f"ACCURACY BY IMAGE CONDITION for {ADAPTER_RUN} (exact match, image-clustered 95% CI)"); print(acc.to_string(index=False))

d["total_gain"] = d.ex_lora_real - d.ex_base_real
d["prior_gain_blank"] = d.ex_lora_blank - d.ex_base_blank
d["benefit_change_blank"] = (d.ex_lora_real - d.ex_lora_blank) - (d.ex_base_real - d.ex_base_blank)
d["benefit_change_other"] = (d.ex_lora_real - d.ex_lora_shuffled) - (d.ex_base_real - d.ex_base_shuffled)
rows = []
for name, mask in subsets:
    r = {"subset": name}
    for col in ("total_gain", "prior_gain_blank", "benefit_change_blank", "benefit_change_other"): r[col], n = ci(col, mask)
    r["n"] = n; rows.append(r)
dec = pd.DataFrame(rows); dec.to_csv(OUT / "gain_decomposition.csv", index=False)
print("\nGAIN DECOMPOSITION (benefit_change_other = change in image benefit measured against a DIFFERENT real image; the more reliable one)"); print(dec.to_string(index=False))

for m in ("base", "lora"): d[f"sens_{m}"] = [float(norm(a) != norm(b)) for a, b in zip(d[f"pr_{m}_real"], d[f"pr_{m}_blank"])]
rows = []
for name, mask in subsets[:3]:
    r = {"subset": name}
    for m in ("base", "lora"): r[f"sensitivity_{m}"], n = ci(f"sens_{m}", mask)
    r["n"] = n; rows.append(r)
print("\nIMAGE SENSITIVITY (answer changes when the image is blanked)"); print(pd.DataFrame(rows).to_string(index=False))
right = d.ex_lora_real == 1
print(f"\nOf the {int(right.sum())} questions this adapter answers correctly with the real image, {d.loc[right, 'ex_lora_blank'].mean():.1%} are also correct with a blank image.")

closed = (d.answer_type == "CLOSED").values
print("\nCLOSED questions: share of answers that are exactly 'yes' or 'no'")
for m in ("base", "lora"):
    for c in ("real", "blank", "shuffled"):
        print(f"  {m:4s} {c:8s} {np.mean([norm(x) in ('yes', 'no') for x in d[f'pr_{m}_{c}'].values[closed]]):.1%}")

tr = pd.DataFrame({"q": [norm(x) for x in train_en["question"]], "a": [norm(x) for x in train_en["answer"]], "t": list(train_en["answer_type"])})
by_q = tr.groupby("q")["a"].agg(lambda s: s.value_counts().index[0]); by_t = tr.groupby("t")["a"].agg(lambda s: s.value_counts().index[0])
d["ex_prior_only"] = [float(by_q.get(norm(q), by_t[t]) == norm(g)) for q, t, g in zip(test_en["question"][:n_eval], d["answer_type"], d["gt"])]
print("\nLOOKUP BASELINE (most common training answer per question text, no image, no model):", ci("ex_prior_only")[0])

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8)); cc = ["real", "shuffled", "blank", "noimage"]; xs = np.arange(4)
for k, (m, lab) in enumerate((("base", "zero-shot"), ("lora", ADAPTER_RUN))): ax[0].bar(xs + (k - 0.5) * 0.4, [d[f"ex_{m}_{c}"].mean() for c in cc], 0.4, label=lab)
ax[0].set_xticks(xs); ax[0].set_xticklabels(cc); ax[0].set_ylim(0, 1); ax[0].legend(fontsize=7); ax[0].set_title("Exact match by image condition")
for k, (nm, mask) in enumerate((("overall", np.ones(len(d), bool)), ("CLOSED", (d.answer_type == "CLOSED").values), ("OPEN", (d.answer_type == "OPEN").values))):
    ax[1].bar(k * 3 + np.arange(2) - 0.2, [d.loc[mask, "benefit_change_blank"].mean(), d.loc[mask, "benefit_change_other"].mean()], 0.4, color=["#b45309", "#1d4ed8"])
ax[1].set_xticks([0, 3, 6]); ax[1].set_xticklabels(["overall", "CLOSED", "OPEN"]); ax[1].axhline(0, color="black", lw=.8)
ax[1].set_title("Change in image benefit: vs blank (orange), vs other image (blue)")
plt.tight_layout(); plt.savefig(OUT / "wp3b_counterfactuals.png", dpi=150); plt.show()
push_folder(OUT, path_in_repo=f"wp3b/{ADAPTER_RUN}", msg=f"wp3b {ADAPTER_RUN}: analysis")

In [ ]:
# CELL G - record results: per-run note + GitHub push into this analysis's OWN folder (skipped in quick-test mode)
import datetime
ov = acc[acc.subset == "overall"].iloc[0]; dv = dec[dec.subset == "overall"].iloc[0]
note = f"""## {datetime.date.today()} - WP3b counterfactuals for {ADAPTER_RUN} (n={n_eval})
- overall exact: base real {ov.base_real} | lora real {ov.lora_real} | lora blank {ov.lora_blank} | lora other-image {ov.lora_shuffled} | lora no-image {ov.lora_noimage}
- gain decomposition (overall): total {dv.total_gain} | gain with blank image {dv.prior_gain_blank} | benefit change vs blank {dv.benefit_change_blank} | vs other image {dv.benefit_change_other}
"""
print(note)
if TEST:
    print("QUICK TEST: skipping GitHub push. Set N_EVAL = None for the real run.")
else:
    dest = REPO / "results" / RUN_TAG; dest.mkdir(parents=True, exist_ok=True)
    sh(f"cp {OUT}/*.csv {OUT}/*.png {dest}/")
    (dest / "NOTE.md").write_text(note)
    url = f"https://{GH_TOKEN}@github.com/{GITHUB_REPO}.git"
    sh(f"cd {REPO} && git add results && git commit -m 'WP3b {ADAPTER_RUN}'; git pull --rebase {url} main; git push {url} HEAD:main")
print("DONE", RUN_TAG)